In [ ]:
import numpy as np

# Reworking matrix formation in hydra

In [ ]:
def gcr_fgmodes(
    vis, w, fgmodes, Nparams, sys_model_past, flags, signal_S, Ninv, f0=None, nproc=1, map_estimate=False,
    verbose=False
):
    """
    Perform the GCR step on all time samples, using parallelisation if
    possible.

    Parameters:
        vis (array_like):
            Array of complex visibilities for a single baseline, of shape
            `(Ntimes, Nfreqs)`.
        w (array_like):
            Array of flags or weights (e.g. 1 for unflagged, 0 for flagged).
        matrices (array_like):
            Array containing precomputed matrices needed by the linear system.
        fgmodes (array_like):
            Foreground mode array, of shape (Nfreqs, Nmodes). This should be
            derived from a PCA decomposition of a model foreground covariance
            matrix or similar.
        fourier_op (array_like):
            Pre-computed Fourier operator.
        f0 (array_like):
            Initial guess for the foreground amplitudes, with shape `(Nmodes,)`.
        nproc (int):
            Number of processes to use for parallelised functions.
        map_estimate (bool):
            Provide the maximum a posteriori sample.
        verbose (bool):
            If True, output basic timing stats about each iteration.

    Returns:
        samples (array_like):
            Array of signal + foreground realisations for each time sample,
            of shape `(Ntimes, Nfreqs + Nmodes)`.
    """
    samples = np.zeros((vis.shape[0], vis.shape[1] + fgmodes.shape[1]), dtype=complex)
    if verbose:
        residuals = np.zeros(vis.shape[0], dtype=float)
        info = np.zeros(vis.shape[0], dtype=float)
    else:
        residuals = None
        info = None
    idxs = np.arange(vis.shape[0])
    
    #Time invariant calculations
    Eh=sp.linalg.sqrtm(E)  
    Nih = np.sqrt(Ninv.diag())
    
    # Run GCR method on each time sample in parallel
    if verbose:
        st = time.time()
    with Pool(nproc) as pool:
        samples, residuals, info = zip(*pool.map(
            lambda idx: gcr_fgmodes_1d_v2(
                idx=idx,
                vis=vis[idx],
                w=w,
                fgmodes=fgmodes,
                f0=f0,
                Nparams=Nparams,
                y=sys_model_past[idx],
                flags=flags,
                E=signal_S,
                Ninv=Ninv,
                Eh=Eh,
                Nih=Nih,
                map_estimate=map_estimate,
                verbose=verbose
            ),
            idxs,
        )
        )
    samples = np.array(samples).reshape((vis.shape[0], -1))
    residuals = np.array(residuals)
    info = np.array(info)

    # Return sample
    if verbose:
        print(f"{time.time() - st:<12.1f}", end="")
        print(f"{info.mean():<8.1f}", end="")
        print(f"{residuals.mean():<12.2e}", end="")
    return samples

In [ ]:
def gcr_fgmodes_1d_v2(idx, vis, w, Nparams, y, flags, E, Ninv, fgmodes, Eh, Nih, f0=None, map_estimate=False, verbose=False,
    multiprocess_seed=912983):

    pid = current_process().pid
    seed = multiprocess_seed + pid*1000 + idx
    np.random.seed(seed)

    Nfreqs, Nmodes = fgmodes.shape
    d = vis.reshape((1, max(Nfreqs, len(vis.T))))

    # Extract precomputed matrices needed by the linear system
    A, Ni, Ai =build_matrices(Nparams, y, flags, E, Ninv, fgmodes)

    if map_estimate:
        oma = np.zeros((Nfreqs, 1), dtype=complex)
        omb = np.zeros((Nfreqs, 1), dtype=complex)
    else:
        # Unit complex Gaussian random realisation
        omi, omj = np.random.randn(Nfreqs, 1), np.random.randn(Nfreqs, 1)
        omk, oml = np.random.randn(Nfreqs, 1), np.random.randn(Nfreqs, 1)
        oma, omb = (omi + 1.0j * omj) / 2**0.5, (omk + 1.0j * oml) / 2**0.5

    # Construct RHS vector
    b = np.zeros((Nfreqs + Nmodes, 1), dtype=complex)
    b[:Nfreqs] = E @ (Ni * w * d) + Eh @ oma + E @ (Nih * omb)
    b[Nfreqs:] = fgmodes.T.conj() @ (Ni * w * d) + (Nih*omb)
    print("Shapes: \nS:",S.shape,
        #   "\ny.conj:",y.conj().shape,
        "\nNi: ",Ni.shape,
        "\nw:",w.shape,
        "\nvis: ",vis.shape,
        "\nSh: ",Sh.shape,
        "\noma: ",oma.shape,
        "\nomb: ",omb.shape,
        "\nNih: ",Nih.shape,
        "\nfgmodes.T.conj(): ",fgmodes.T.conj().shape,
        "\nfgmodes: ",fgmodes.shape)
    # Run CG solver, preconditioned by M=Ai
    x0 = None
    if f0 is not None:
        x0 = np.concatenate((np.zeros(Nfreqs, dtype=complex), f0))
    xsoln, info = sp.sparse.linalg.cgs(A, b, maxiter=int(1e5), x0=x0, M=Ai)
    if verbose:
        residual = np.abs(A @ xsoln - b[:, 0]).mean()
    else:
        residual = None

    # Return solution vector
    return xsoln, residual, info

In [ ]:
def build_matrices(Nparams, y, flags, E, Ninv, fgmodes):
    """
    Calculate matrices and build A in Ax=b for the GCR step.
    
    Parameters:
        Nparams (int):
            Number of model parameters.
        flags (array_like):
            Array of flags (1 for unflagged, 0 for flagged), with shape 
            `(Nfreqs,)`.
        signal_S (array_like):
            Current value of the EoR signal frequency-frequency covariance.
        Ninv (array_like):
            Inverse noise variance matrix. This can either have shape
            `(Ntimes, Nfreqs, Nfreqs)`, one for each time, or can be a common
            one for all times with shape `(Nfreqs, Nfreqs)`.
        fgmodes (array_like):
            Foreground mode array, of shape (Nfreqs, Nmodes). This should be
            derived from a PCA decomposition of a model foreground covariance
            matrix or similar.
    
    Returns:
        matrices (list of array_like):
            List containing necessary GCR operators (`matrices[0]`) and the
            linear operator A in the GCR Ax=b solve step.
    """
    Nfreqs = E.shape[0]
    
    # Construct necessary operators for GCR
    inner_prod= (y.conj().T * Ninv.diagonal() *  y)
    Ni_flagged = flags.T * (inner_prod) * flags  # Ni # FIXME
    # print("build_matrices shape check:\n y: {}\n flags: {}\n E: {}\n Ninv: {}\n fgmodes: {}\n Ni_flagged: {}".format(y.shape,flags.shape,E.shape,Ninv.shape,fgmodes.shape, Ni_flagged.shape))
    
    # Construct operator matrix
    A = np.zeros((Nparams, Nparams), dtype=complex)
    # print(np.shape(A[Nfreqs:, Nfreqs:]))
    A[:Nfreqs, :Nfreqs] = np.eye(Nfreqs) + E * Ni_flagged[:,np.newaxis]  # 1 + E @ y.dag * Ni * y
    A[:Nfreqs, Nfreqs:] = E @ (Ni_flagged[:,np.newaxis] * fgmodes)
    A[Nfreqs:, :Nfreqs] = (fgmodes.conj() * Ni_flagged[:,np.newaxis]).T
    A[Nfreqs:, Nfreqs:] = fgmodes.T.conj() @ (Ni_flagged[:,np.newaxis] * fgmodes)

    A_pinv = np.linalg.pinv(A)  # pseudo-inverse, to be used as a preconditioner
    
    return A, Ni_flagged, A_pinv

Version 2 of systematics-as-gain build_matrices() after doing the maths

In [ ]:
def build_matrices(Nparams, y, flags, E, Ninv, fgmodes):
    """
    Calculate matrices and build A in Ax=b for the GCR step.
    
    Parameters:
        Nparams (int):
            Number of model parameters.
        flags (array_like):
            Array of flags (1 for unflagged, 0 for flagged), with shape 
            `(Nfreqs,)`.
        signal_S (array_like):
            Current value of the EoR signal frequency-frequency covariance.
        Ninv (array_like):
            Inverse noise variance matrix. This can either have shape
            `(Ntimes, Nfreqs, Nfreqs)`, one for each time, or can be a common
            one for all times with shape `(Nfreqs, Nfreqs)`.
        fgmodes (array_like):
            Foreground mode array, of shape (Nfreqs, Nmodes). This should be
            derived from a PCA decomposition of a model foreground covariance
            matrix or similar.
    
    Returns:
        matrices (list of array_like):
            List containing necessary GCR operators (`matrices[0]`) and the
            linear operator A in the GCR Ax=b solve step.
    """
    Nfreqs = E.shape[0]
    
    E_inv=np.linalg.inv(E) #FIXME
    # G_inv=np.linalg.inv(fgmodes) #FIXME
    
    # Construct necessary operators for GCR
    inner_prod= (y.conj().T * Ninv.diagonal() *  y)
    Ni_flagged = flags.T * (inner_prod) * flags  # Ni # FIXME
    print("build_matrices shape check:\n y: {}\n flags: {}\n signal_S: {}\n Ninv: {}\n fgmodes: {}\n Ni_flagged: {}".format(y.shape,flags.shape,E.shape,Ninv.shape,fgmodes.shape, Ni_flagged.shape))
    
    # Construct operator matrix
    A = np.zeros((Nparams, Nparams), dtype=complex)
    A[:Nfreqs, :Nfreqs] = y.conj()[:,np.newaxis] * E_inv * y[:,np.newaxis] + Ni_flagged  # A11: y.dag E^-1 y + y.dag * Ni * y
    A[:Nfreqs, Nfreqs:] = Ni_flagged[:,np.newaxis] * fgmodes # A12: y.dag * Ni * y * G
    A[Nfreqs:, :Nfreqs] = (fgmodes.conj() * Ni_flagged[:,np.newaxis]).T #A21: G.dag * y.dag * Ni * y
    A[Nfreqs:, Nfreqs:] = fgmodes.T.conj() @ (Ni_flagged[:,np.newaxis] * fgmodes) #A22: y.dag * G^-1 *y + G.dag * y.dag * Ni * y * G (y.conj()[:,np.newaxis] * G_inv * y[:,np.newaxis] + )

    A_pinv = np.linalg.pinv(A)  # pseudo-inverse, to be used as a preconditioner
    
    return A, Ni_flagged, A_pinv